In [5]:
"""
prepare_geodata.py
==================
Converts raw ONS boundary Shapefiles into optimised GeoJSON files
for the MSA interactive map.

Run from project root:
  python scripts/prepare_geodata.py
"""

import geopandas as gpd
import pandas as pd
import os

# ── Paths ─────────────────────────────────────────────────────────────────
BASE = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"

COUNTRIES_SHP = f"{BASE}/data/raw/boundaries/Countries_December_2025_Boundaries_UK_BGC_1207682485364452102/CTRY_DEC_2025_UK_BGC.shp"
CA_SHP        = f"{BASE}/data/raw/boundaries/Combined_Authorities_December_2025_Boundaries_EN_BGC_8767275729874946565/CAUTH_DEC_2025_EN_BGC.shp"
LAD_SHP       = f"{BASE}/data/raw/boundaries/Local_Authority_Districts_DEC_2025_Boundaries_UK_BUC_-6435411136043329150/LAD_DEC_2025_UK_BUC.shp"
LOOKUP_CSV    = f"{BASE}/data/raw/lookups/Local_Authority_District_to_Combined_Authority_(May_2025)_Lookup_in_EN.csv"

PROCESSED = f"{BASE}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# ── In-scope MSA definitions ──────────────────────────────────────────────

IN_SCOPE_CODES = [
    "E47000001",  # Greater Manchester       — Trailblazer
    "E47000002",  # South Yorkshire          — Level 3
    "E47000003",  # West Yorkshire           — Level 3
    "E47000004",  # Liverpool City Region    — Level 3
    "E47000006",  # Tees Valley              — Level 3
    "E47000007",  # West Midlands            — Trailblazer
    "E47000008",  # Cambridgeshire & Pboro   — Level 3
    "E47000009",  # West of England          — Level 3
    "E47000012",  # York & North Yorkshire   — Level 3
    "E47000013",  # East Midlands            — Level 3
    "E47000014",  # North East               — Level 4
    "E47000016",  # Hull & East Yorkshire    — Level 3
    "E47000017",  # Greater Lincolnshire     — Level 3
    "E47000018",  # Lancashire               — Level 3
]

DEAL_LEVELS = {
    "E47000001": "trailblazer",
    "E47000007": "trailblazer",
    "E47000014": "level_4",
    "E47000002": "level_3",
    "E47000003": "level_3",
    "E47000004": "level_3",
    "E47000006": "level_3",
    "E47000008": "level_3",
    "E47000009": "level_3",
    "E47000012": "level_3",
    "E47000013": "level_3",
    "E47000016": "level_3",
    "E47000017": "level_3",
    "E47000018": "level_3",
}

SHORT_NAMES = {
    "E47000001": "Greater Manchester",
    "E47000002": "South Yorkshire",
    "E47000003": "West Yorkshire",
    "E47000004": "Liverpool City Region",
    "E47000006": "Tees Valley",
    "E47000007": "West Midlands",
    "E47000008": "Cambridgeshire & Peterborough",
    "E47000009": "West of England",
    "E47000012": "York & North Yorkshire",
    "E47000013": "East Midlands",
    "E47000014": "North East",
    "E47000016": "Hull & East Yorkshire",
    "E47000017": "Greater Lincolnshire",
    "E47000018": "Lancashire",
    "E47000015": "Devon & Torbay",
}

# ── 1. Countries ──────────────────────────────────────────────────────────
print("── 1. Processing countries ──────────────────────────────────────")
countries = gpd.read_file(COUNTRIES_SHP)
print(f"   Read {len(countries)} features, CRS: {countries.crs}")

countries = countries.to_crs(epsg=4326)
countries = countries[["CTRY25CD", "CTRY25NM", "geometry"]].copy()
countries["is_england"] = countries["CTRY25CD"] == "E92000001"

countries.to_file(f"{PROCESSED}/countries.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/countries.geojson") / 1024
print(f"   Saved countries.geojson — {len(countries)} features, {size:.0f} KB")


# ── 2. Combined Authorities ───────────────────────────────────────────────
print("\n── 2. Processing Combined Authorities ───────────────────────────")
ca = gpd.read_file(CA_SHP)
print(f"   Read {len(ca)} features, CRS: {ca.crs}")

ca = ca.to_crs(epsg=4326)
ca["geometry"] = ca["geometry"].simplify(0.005, preserve_topology=True)
ca = ca[["CAUTH25CD", "CAUTH25NM", "geometry"]].copy()

ca["status"]     = ca["CAUTH25CD"].apply(
    lambda x: "in_scope" if x in IN_SCOPE_CODES else "out_of_scope"
)
ca["deal_level"] = ca["CAUTH25CD"].map(DEAL_LEVELS).fillna("not_in_scope")
ca["short_name"] = ca["CAUTH25CD"].map(SHORT_NAMES).fillna(ca["CAUTH25NM"])

print("\n   CA features:")
for _, row in ca.iterrows():
    print(f"   {row['CAUTH25CD']}  {row['deal_level']:15}  {row['CAUTH25NM']}")

ca.to_file(f"{PROCESSED}/ca.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/ca.geojson") / 1024
print(f"\n   Saved ca.geojson — {len(ca)} features, {size:.0f} KB")
print(f"   In-scope: {ca[ca['status']=='in_scope'].shape[0]}  |  "
      f"Out of scope: {ca[ca['status']=='out_of_scope'].shape[0]}")


# ── 3. Local Authority Districts ──────────────────────────────────────────
print("\n── 3. Processing LADs ───────────────────────────────────────────")
lad = gpd.read_file(LAD_SHP)
print(f"   Read {len(lad)} features, CRS: {lad.crs}")

lad = lad[lad["LAD25CD"].str.startswith("E")].copy()
print(f"   After filtering to England: {len(lad)} features")

lad = lad.to_crs(epsg=4326)
lad["geometry"] = lad["geometry"].simplify(0.003, preserve_topology=True)
lad = lad[["LAD25CD", "LAD25NM", "geometry"]].copy()


# ── 4. Join lookup ────────────────────────────────────────────────────────
print("\n── 4. Joining LAD to CA lookup ──────────────────────────────────")
lookup = pd.read_csv(LOOKUP_CSV)
print(f"   Lookup columns: {lookup.columns.tolist()}")
print(f"   Lookup rows:    {len(lookup)}")

try:
    lookup_slim = lookup[["LAD25CD", "CAUTH25CD", "CAUTH25NM"]].copy()
    lookup_slim.columns = ["LAD25CD", "ca_gss_code", "ca_name"]
except KeyError as e:
    print(f"\n   ERROR: Column not found in lookup CSV: {e}")
    print(f"   Available columns: {lookup.columns.tolist()}")
    raise

lad = lad.merge(lookup_slim, on="LAD25CD", how="left")
lad["ca_gss_code"]   = lad["ca_gss_code"].fillna("none")
lad["ca_name"]       = lad["ca_name"].fillna("No Combined Authority")
lad["ca_deal_level"] = lad["ca_gss_code"].map(DEAL_LEVELS).fillna("none")

matched   = (lad["ca_gss_code"] != "none").sum()
unmatched = (lad["ca_gss_code"] == "none").sum()
print(f"   LADs matched to a CA:        {matched}")
print(f"   LADs with no CA (expected):  {unmatched}")

lad.to_file(f"{PROCESSED}/lad.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/lad.geojson") / 1024
print(f"   Saved lad.geojson — {len(lad)} features, {size:.0f} KB")


# ── 5. Summary ────────────────────────────────────────────────────────────
print("\n── Summary ──────────────────────────────────────────────────────")
for fname in ["countries.geojson", "ca.geojson", "lad.geojson"]:
    path = f"{PROCESSED}/{fname}"
    size = os.path.getsize(path) / 1024
    print(f"   {fname:25}  {size:6.0f} KB")

print("\n   All done. Files ready in data/processed/")
print("   Next step: open VS Code and start the map build.")

── 1. Processing countries ──────────────────────────────────────
   Read 4 features, CRS: EPSG:27700
   Saved countries.geojson — 4 features, 14700 KB

── 2. Processing Combined Authorities ───────────────────────────
   Read 15 features, CRS: EPSG:27700

   CA features:
   E47000001  trailblazer      Greater Manchester
   E47000002  level_3          South Yorkshire
   E47000003  level_3          West Yorkshire
   E47000004  level_3          Liverpool City Region
   E47000006  level_3          Tees Valley
   E47000007  trailblazer      West Midlands
   E47000008  level_3          Cambridgeshire and Peterborough
   E47000009  level_3          West of England
   E47000012  level_3          York and North Yorkshire
   E47000013  level_3          East Midlands
   E47000014  level_4          North East
   E47000015  not_in_scope     Devon and Torbay
   E47000016  level_3          Hull and East Yorkshire
   E47000017  level_3          Greater Lincolnshire
   E47000018  level_3          Lanc

In [6]:
import geopandas as gpd
import os

BASE = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"

countries = gpd.read_file(f"{BASE}/data/raw/boundaries/Countries_December_2025_Boundaries_UK_BGC_1207682485364452102/CTRY_DEC_2025_UK_BGC.shp")
countries = countries.to_crs(epsg=4326)
countries["geometry"] = countries["geometry"].simplify(0.01, preserve_topology=True)
countries = countries[["CTRY25CD", "CTRY25NM", "geometry"]].copy()
countries["is_england"] = countries["CTRY25CD"] == "E92000001"
countries.to_file(f"{BASE}/data/processed/countries.geojson", driver="GeoJSON")

size = os.path.getsize(f"{BASE}/data/processed/countries.geojson") / 1024
print(f"countries.geojson: {size:.0f} KB")

countries.geojson: 1277 KB


In [8]:
lookup_lsoa = pd.read_csv("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map/data/raw/lookups/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv")


# Drop duplicates to get one row per LSOA
lsoa_lad = lookup_lsoa[[
    "OA21CD", "LSOA21CD", "LSOA21NM",
    "MSOA21CD", "MSOA21NM",
    "LAD22CD", "LAD22NM"
]].drop_duplicates(subset="LSOA21CD")

print(f"Unique LSOAs: {len(lsoa_lad)}")

# Try joining to your LAD25 → CA lookup
ca_lookup = pd.read_csv("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map/data/raw/lookups/Local_Authority_District_to_Combined_Authority_(May_2025)_Lookup_in_EN.csv")

merged = lsoa_lad.merge(
    ca_lookup[["LAD25CD", "CAUTH25CD", "CAUTH25NM"]],
    left_on="LAD22CD",
    right_on="LAD25CD",
    how="left"
)

matched   = merged["CAUTH25CD"].notna().sum()
unmatched = merged["CAUTH25CD"].isna().sum()
print(f"Matched:   {matched}")
print(f"Unmatched: {unmatched}")

# Show which LAD22 codes didn't match — paste this output
unmatched_lads = merged[merged["CAUTH25CD"].isna()]["LAD22CD"].unique()
print(f"\nUnmatched LAD22 codes ({len(unmatched_lads)}):")
print(unmatched_lads)

Unique LSOAs: 35672
Matched:   12948
Unmatched: 22724

Unmatched LAD22 codes (228):
['E06000007' 'E06000016' 'E06000017' 'E06000019' 'E06000020' 'E06000021'
 'E06000024' 'E06000026' 'E06000030' 'E06000032' 'E06000033' 'E06000034'
 'E06000035' 'E06000036' 'E06000037' 'E06000038' 'E06000039' 'E06000040'
 'E06000041' 'E06000042' 'E06000043' 'E06000044' 'E06000045' 'E06000046'
 'E06000049' 'E06000050' 'E06000051' 'E06000052' 'E06000054' 'E06000053'
 'E06000055' 'E06000056' 'E06000058' 'E06000059' 'E06000060' 'E06000061'
 'E06000062' 'E07000026' 'E07000027' 'E07000028' 'E07000029' 'E07000030'
 'E07000031' 'E07000061' 'E07000062' 'E07000063' 'E07000064' 'E07000065'
 'E07000066' 'E07000067' 'E07000068' 'E07000069' 'E07000070' 'E07000071'
 'E07000072' 'E07000073' 'E07000074' 'E07000075' 'E07000076' 'E07000077'
 'E07000078' 'E07000079' 'E07000080' 'E07000081' 'E07000082' 'E07000083'
 'E07000084' 'E07000085' 'E07000086' 'E07000087' 'E07000088' 'E07000089'
 'E07000090' 'E07000091' 'E07000092' 'E0

/var/folders/0x/4hyvr0bx34dbw7d0554jq_1c0000gp/T/ipykernel_19277/1617530121.py:1: DtypeWarning: Columns (3,6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup_lsoa = pd.read_csv("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map/data/raw/lookups/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv")


In [9]:

# Filter to England only first
england_lsoas = lsoa_lad[lsoa_lad["LAD22CD"].str.startswith("E")].copy()
wales_lsoas   = lsoa_lad[lsoa_lad["LAD22CD"].str.startswith("W")].copy()

print(f"England LSOAs: {len(england_lsoas)}")
print(f"Wales LSOAs:   {len(wales_lsoas)}")

# Re-run merge on England only
merged_eng = england_lsoas.merge(
    ca_lookup[["LAD25CD", "CAUTH25CD", "CAUTH25NM"]],
    left_on="LAD22CD",
    right_on="LAD25CD",
    how="left"
)

in_ca     = merged_eng["CAUTH25CD"].notna().sum()
not_in_ca = merged_eng["CAUTH25CD"].isna().sum()

print(f"\nEnglish LSOAs inside a CA:    {in_ca}")
print(f"English LSOAs outside any CA: {not_in_ca}")
print(f"(outside CA = rest of England, expected to be majority)")

# Check if any in-scope MSA LAD codes appear in unmatched
in_scope_lads = ca_lookup["LAD25CD"].tolist()
unmatched_eng = merged_eng[merged_eng["CAUTH25CD"].isna()]["LAD22CD"].unique()
false_misses  = [c for c in unmatched_eng if c in in_scope_lads]
print(f"\nIn-scope LAD codes that failed to match: {len(false_misses)}")
if false_misses:
    print(false_misses)

England LSOAs: 33755
Wales LSOAs:   1917

English LSOAs inside a CA:    12948
English LSOAs outside any CA: 20807
(outside CA = rest of England, expected to be majority)

In-scope LAD codes that failed to match: 0


In [11]:
"""
prepare_geodata.py
==================
Converts raw ONS boundary Shapefiles into optimised GeoJSON files
for the MSA interactive map.

Run from project root:
  python scripts/prepare_geodata.py

Outputs:
  data/processed/countries.geojson
  data/processed/ca.geojson
  data/processed/lad.geojson
  data/processed/msoa.geojson
  data/processed/lsoa.geojson
"""

import geopandas as gpd
import pandas as pd
import os

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"

# Raw boundary shapefiles
COUNTRIES_SHP = f"{BASE}/data/raw/boundaries/Countries_December_2025_Boundaries_UK_BGC_1207682485364452102/CTRY_DEC_2025_UK_BGC.shp"
CA_SHP        = f"{BASE}/data/raw/boundaries/Combined_Authorities_December_2025_Boundaries_EN_BGC_8767275729874946565/CAUTH_DEC_2025_EN_BGC.shp"
LAD_SHP       = f"{BASE}/data/raw/boundaries/Local_Authority_Districts_DEC_2025_Boundaries_UK_BUC_-6435411136043329150/LAD_DEC_2025_UK_BUC.shp"
MSOA_SHP      = f"{BASE}/data/raw/boundaries/Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_1489062760399791320/MSOA_2021_EW_BGC_V3.shp"
LSOA_SHP      = f"{BASE}/data/raw/boundaries/Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_-7203918579177758597/LSOA_2021_EW_BGC_V5.shp"

# Raw lookups
LOOKUP_CSV    = f"{BASE}/data/raw/lookups/Local_Authority_District_to_Combined_Authority_(May_2025)_Lookup_in_EN.csv"
OA_LOOKUP_CSV = f"{BASE}/data/raw/lookups/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv"

PROCESSED = f"{BASE}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# ── In-scope MSA definitions ───────────────────────────────────────────────
IN_SCOPE_CODES = [
    "E47000001",  # Greater Manchester       — Trailblazer
    "E47000002",  # South Yorkshire          — Level 3
    "E47000003",  # West Yorkshire           — Level 3
    "E47000004",  # Liverpool City Region    — Level 3
    "E47000006",  # Tees Valley              — Level 3
    "E47000007",  # West Midlands            — Trailblazer
    "E47000008",  # Cambridgeshire & Pboro   — Level 3
    "E47000009",  # West of England          — Level 3
    "E47000012",  # York & North Yorkshire   — Level 3
    "E47000013",  # East Midlands            — Level 3
    "E47000014",  # North East               — Level 4
    "E47000016",  # Hull & East Yorkshire    — Level 3
    "E47000017",  # Greater Lincolnshire     — Level 3
    "E47000018",  # Lancashire               — Level 3
]

DEAL_LEVELS = {
    "E47000001": "trailblazer",
    "E47000007": "trailblazer",
    "E47000014": "level_4",
    "E47000002": "level_3",
    "E47000003": "level_3",
    "E47000004": "level_3",
    "E47000006": "level_3",
    "E47000008": "level_3",
    "E47000009": "level_3",
    "E47000012": "level_3",
    "E47000013": "level_3",
    "E47000016": "level_3",
    "E47000017": "level_3",
    "E47000018": "level_3",
}

SHORT_NAMES = {
    "E47000001": "Greater Manchester",
    "E47000002": "South Yorkshire",
    "E47000003": "West Yorkshire",
    "E47000004": "Liverpool City Region",
    "E47000006": "Tees Valley",
    "E47000007": "West Midlands",
    "E47000008": "Cambridgeshire & Peterborough",
    "E47000009": "West of England",
    "E47000012": "York & North Yorkshire",
    "E47000013": "East Midlands",
    "E47000014": "North East",
    "E47000016": "Hull & East Yorkshire",
    "E47000017": "Greater Lincolnshire",
    "E47000018": "Lancashire",
    "E47000015": "Devon & Torbay",
}

# North Yorkshire old district codes — reorganised April 2023
NORTH_YORKS_OLD = [
    "E07000163",  # Craven
    "E07000164",  # Hambleton
    "E07000165",  # Harrogate
    "E07000166",  # Richmondshire
    "E07000167",  # Ryedale
    "E07000168",  # Scarborough
    "E07000169",  # Selby
]

# ── Load shared lookups once ───────────────────────────────────────────────
print("── Loading shared lookup files ──────────────────────────────────")
ca_lookup = pd.read_csv(LOOKUP_CSV)
print(f"   CA lookup rows: {len(ca_lookup)}")

oa_lookup = pd.read_csv(OA_LOOKUP_CSV, low_memory=False)
print(f"   OA lookup rows: {len(oa_lookup)}")

# ── 1. Countries ───────────────────────────────────────────────────────────
print("\n── 1. Processing countries ──────────────────────────────────────")
countries = gpd.read_file(COUNTRIES_SHP)
print(f"   Read {len(countries)} features, CRS: {countries.crs}")

countries = countries.to_crs(epsg=4326)
countries["geometry"] = countries["geometry"].simplify(0.01, preserve_topology=True)
countries = countries[["CTRY25CD", "CTRY25NM", "geometry"]].copy()
countries["is_england"] = countries["CTRY25CD"] == "E92000001"

countries.to_file(f"{PROCESSED}/countries.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/countries.geojson") / 1024
print(f"   Saved countries.geojson — {len(countries)} features, {size:.0f} KB")


# ── 2. Combined Authorities ────────────────────────────────────────────────
print("\n── 2. Processing Combined Authorities ───────────────────────────")
ca = gpd.read_file(CA_SHP)
print(f"   Read {len(ca)} features, CRS: {ca.crs}")

ca = ca.to_crs(epsg=4326)
ca["geometry"] = ca["geometry"].simplify(0.005, preserve_topology=True)
ca = ca[["CAUTH25CD", "CAUTH25NM", "geometry"]].copy()

ca["status"]     = ca["CAUTH25CD"].apply(lambda x: "in_scope" if x in IN_SCOPE_CODES else "out_of_scope")
ca["deal_level"] = ca["CAUTH25CD"].map(DEAL_LEVELS).fillna("not_in_scope")
ca["short_name"] = ca["CAUTH25CD"].map(SHORT_NAMES).fillna(ca["CAUTH25NM"])

print("\n   CA features:")
for _, row in ca.iterrows():
    print(f"   {row['CAUTH25CD']}  {row['deal_level']:15}  {row['CAUTH25NM']}")

ca.to_file(f"{PROCESSED}/ca.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/ca.geojson") / 1024
print(f"\n   Saved ca.geojson — {len(ca)} features, {size:.0f} KB")
print(f"   In-scope: {ca[ca['status']=='in_scope'].shape[0]}  |  Out of scope: {ca[ca['status']=='out_of_scope'].shape[0]}")


# ── 3. Local Authority Districts ───────────────────────────────────────────
print("\n── 3. Processing LADs ───────────────────────────────────────────")
lad = gpd.read_file(LAD_SHP)
print(f"   Read {len(lad)} features, CRS: {lad.crs}")

lad = lad[lad["LAD25CD"].str.startswith("E")].copy()
print(f"   After filtering to England: {len(lad)} features")

lad = lad.to_crs(epsg=4326)
lad["geometry"] = lad["geometry"].simplify(0.003, preserve_topology=True)
lad = lad[["LAD25CD", "LAD25NM", "geometry"]].copy()

lookup_slim = ca_lookup[["LAD25CD", "CAUTH25CD", "CAUTH25NM"]].copy()
lookup_slim.columns = ["LAD25CD", "ca_gss_code", "ca_name"]

lad = lad.merge(lookup_slim, on="LAD25CD", how="left")
lad["ca_gss_code"]   = lad["ca_gss_code"].fillna("none")
lad["ca_name"]       = lad["ca_name"].fillna("No Combined Authority")
lad["ca_deal_level"] = lad["ca_gss_code"].map(DEAL_LEVELS).fillna("none")

matched   = (lad["ca_gss_code"] != "none").sum()
unmatched = (lad["ca_gss_code"] == "none").sum()
print(f"   LADs matched to a CA:        {matched}")
print(f"   LADs with no CA (expected):  {unmatched}")

lad.to_file(f"{PROCESSED}/lad.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/lad.geojson") / 1024
print(f"   Saved lad.geojson — {len(lad)} features, {size:.0f} KB")


# ── 4. MSOAs ───────────────────────────────────────────────────────────────
print("\n── 4. Processing MSOAs ──────────────────────────────────────────")
msoa = gpd.read_file(MSOA_SHP)
print(f"   Read {len(msoa)} features, CRS: {msoa.crs}")

msoa = msoa[msoa["MSOA21CD"].str.startswith("E")].copy()
print(f"   After filtering to England: {len(msoa)} features")

msoa = msoa.to_crs(epsg=4326)
msoa["geometry"] = msoa["geometry"].simplify(0.001, preserve_topology=True)
msoa = msoa[["MSOA21CD", "MSOA21NM", "geometry"]].copy()

# Build MSOA → LAD → CA lookup
msoa_lad = oa_lookup[["MSOA21CD", "LAD22CD", "LAD22NM"]].drop_duplicates(subset="MSOA21CD")
msoa_lad = msoa_lad.merge(
    ca_lookup[["LAD25CD", "CAUTH25CD", "CAUTH25NM"]],
    left_on="LAD22CD",
    right_on="LAD25CD",
    how="left"
)

# North Yorkshire fix
mask = msoa_lad["LAD22CD"].isin(NORTH_YORKS_OLD)
msoa_lad.loc[mask, "CAUTH25CD"] = "E47000012"
msoa_lad.loc[mask, "CAUTH25NM"] = "York and North Yorkshire"
msoa_lad.loc[mask, "LAD22NM"]   = "North Yorkshire"
print(f"   North Yorkshire MSOAs patched: {mask.sum()}")

msoa = msoa.merge(
    msoa_lad[["MSOA21CD", "LAD22CD", "LAD22NM", "CAUTH25CD", "CAUTH25NM"]],
    on="MSOA21CD",
    how="left"
)
msoa["CAUTH25CD"]    = msoa["CAUTH25CD"].fillna("none")
msoa["CAUTH25NM"]    = msoa["CAUTH25NM"].fillna("No Combined Authority")
msoa["ca_deal_level"] = msoa["CAUTH25CD"].map(DEAL_LEVELS).fillna("none")

msoa.to_file(f"{PROCESSED}/msoa.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/msoa.geojson") / 1024
print(f"   Saved msoa.geojson — {len(msoa)} features, {size:.0f} KB")
in_ca = (msoa["CAUTH25CD"] != "none").sum()
print(f"   MSOAs inside a CA: {in_ca}  |  Outside: {len(msoa) - in_ca}")


# ── 5. LSOAs ───────────────────────────────────────────────────────────────
print("\n── 5. Processing LSOAs ──────────────────────────────────────────")
lsoa = gpd.read_file(LSOA_SHP)
print(f"   Read {len(lsoa)} features, CRS: {lsoa.crs}")

lsoa = lsoa[lsoa["LSOA21CD"].str.startswith("E")].copy()
print(f"   After filtering to England: {len(lsoa)} features")

lsoa = lsoa.to_crs(epsg=4326)
lsoa["geometry"] = lsoa["geometry"].simplify(0.0005, preserve_topology=True)
lsoa = lsoa[["LSOA21CD", "LSOA21NM", "geometry"]].copy()

# Build LSOA → LAD → CA lookup
lsoa_lad = oa_lookup[["LSOA21CD", "MSOA21CD", "LAD22CD", "LAD22NM"]].drop_duplicates(subset="LSOA21CD")
lsoa_lad = lsoa_lad.merge(
    ca_lookup[["LAD25CD", "CAUTH25CD", "CAUTH25NM"]],
    left_on="LAD22CD",
    right_on="LAD25CD",
    how="left"
)

# North Yorkshire fix
mask = lsoa_lad["LAD22CD"].isin(NORTH_YORKS_OLD)
lsoa_lad.loc[mask, "CAUTH25CD"] = "E47000012"
lsoa_lad.loc[mask, "CAUTH25NM"] = "York and North Yorkshire"
lsoa_lad.loc[mask, "LAD22NM"]   = "North Yorkshire"
print(f"   North Yorkshire LSOAs patched: {mask.sum()}")

lsoa = lsoa.merge(
    lsoa_lad[["LSOA21CD", "MSOA21CD", "LAD22CD", "LAD22NM", "CAUTH25CD", "CAUTH25NM"]],
    on="LSOA21CD",
    how="left"
)
lsoa["CAUTH25CD"]    = lsoa["CAUTH25CD"].fillna("none")
lsoa["CAUTH25NM"]    = lsoa["CAUTH25NM"].fillna("No Combined Authority")
lsoa["ca_deal_level"] = lsoa["CAUTH25CD"].map(DEAL_LEVELS).fillna("none")

lsoa.to_file(f"{PROCESSED}/lsoa.geojson", driver="GeoJSON")
size = os.path.getsize(f"{PROCESSED}/lsoa.geojson") / 1024
print(f"   Saved lsoa.geojson — {len(lsoa)} features, {size:.0f} KB")
in_ca = (lsoa["CAUTH25CD"] != "none").sum()
print(f"   LSOAs inside a CA: {in_ca}  |  Outside: {len(lsoa) - in_ca}")


# ── Summary ────────────────────────────────────────────────────────────────
print("\n── Summary ──────────────────────────────────────────────────────")
for fname in ["countries.geojson", "ca.geojson", "lad.geojson", "msoa.geojson", "lsoa.geojson"]:
    path = f"{PROCESSED}/{fname}"
    size = os.path.getsize(path) / 1024
    print(f"   {fname:25}  {size:6.0f} KB")

print("\n   All done. Files ready in data/processed/")

── Loading shared lookup files ──────────────────────────────────
   CA lookup rows: 106
   OA lookup rows: 188880

── 1. Processing countries ──────────────────────────────────────
   Read 4 features, CRS: EPSG:27700
   Saved countries.geojson — 4 features, 1277 KB

── 2. Processing Combined Authorities ───────────────────────────
   Read 15 features, CRS: EPSG:27700

   CA features:
   E47000001  trailblazer      Greater Manchester
   E47000002  level_3          South Yorkshire
   E47000003  level_3          West Yorkshire
   E47000004  level_3          Liverpool City Region
   E47000006  level_3          Tees Valley
   E47000007  trailblazer      West Midlands
   E47000008  level_3          Cambridgeshire and Peterborough
   E47000009  level_3          West of England
   E47000012  level_3          York and North Yorkshire
   E47000013  level_3          East Midlands
   E47000014  level_4          North East
   E47000015  not_in_scope     Devon and Torbay
   E47000016  level_3      